In [ ]:
from pathlib import Path
import json
import os

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

SMART_DIR = BASE_DIR / "data" / "SMART-OM"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("SMART_DIR existe:", SMART_DIR.exists())
print("SMART_DIR:", SMART_DIR)

In [ ]:
for item in SMART_DIR.iterdir():
    if item.is_dir():
        print("[DIR] ", item.name)
    else:
        print("[FILE]", item.name)

In [ ]:
carpetas_diagnostico = [
    "01. Normal",
    "02. Variation from normal",
    "03. OPMD",
    "04. Oral Cancer"
]

extensiones_img = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]

resumen = []

for nombre_carpeta in carpetas_diagnostico:
    carpeta = SMART_DIR / nombre_carpeta
    
    imagenes = []
    for ext in extensiones_img:
        imagenes.extend(list(carpeta.rglob(ext)))
    
    resumen.append({
        "carpeta": nombre_carpeta,
        "num_imagenes": len(imagenes)
    })

df_resumen = pd.DataFrame(resumen)
display(df_resumen)

csv_path = TABLES_DIR / "smart_om_resumen_imagenes.csv"
df_resumen.to_csv(csv_path, index=False)

print("Resumen guardado en:", csv_path)

In [ ]:
extensiones_anotacion = [
    "*.json", "*.csv", "*.xlsx", "*.txt", "*.xml"
]

archivos_anotacion = []

for ext in extensiones_anotacion:
    archivos_anotacion.extend(list(SMART_DIR.rglob(ext)))

print("Archivos de anotación/metadatos encontrados:", len(archivos_anotacion))

for f in archivos_anotacion[:100]:
    print(f.relative_to(SMART_DIR))

In [ ]:
info_archivos = []

for f in archivos_anotacion:
    info_archivos.append({
        "ruta": str(f.relative_to(SMART_DIR)),
        "extension": f.suffix,
        "tamano_kb": f.stat().st_size / 1024
    })

df_archivos = pd.DataFrame(info_archivos)
df_archivos = df_archivos.sort_values("tamano_kb", ascending=False)

display(df_archivos.head(30))

In [ ]:
todas_imagenes = []

for nombre_carpeta in carpetas_diagnostico:
    carpeta = SMART_DIR / nombre_carpeta
    
    for ext in extensiones_img:
        for img_path in carpeta.rglob(ext):
            todas_imagenes.append({
                "path": img_path,
                "clase": nombre_carpeta
            })

df_imgs = pd.DataFrame(todas_imagenes)

print("Total imágenes encontradas:", len(df_imgs))
display(df_imgs.head())

In [ ]:
muestras = df_imgs.sample(n=min(8, len(df_imgs)), random_state=7)

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    img = cv2.imread(str(fila["path"]), cv2.IMREAD_COLOR)
    
    if img is None:
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.title(fila["clase"], fontsize=9)
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "smart_om_ejemplos_imagenes.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)